# `Line_Candidates` Stereo Reconstruction and Reco_Tree Validation

This notebook explores the feasibility of reconstructing approximate 3D `(x, y, z)` points from U and V projections in the `Line_Candidates` tree, and validates the results against the fully reconstructed track hits in `Reco_Tree`.

In [ ]:
import uproot
import awkward as ak
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "figure.dpi": 100,
})

print("Imports successful! ✓")

## 1. Open the ROOT File

In [ ]:
file_path = Path("../Cut2.root")

if not file_path.exists():
    raise FileNotFoundError(f"Could not find {file_path.resolve()}")

f = uproot.open(file_path)
print(f"Opened: {file_path.resolve()}")

## 2. Load the Line Candidates and Reconstructed Trees

In [ ]:
t_lc = f["Line_Candidates;2"]
t_rc = f["Reco_Tree;2"]

# Load Line Candidates branches
lc_branches = [
    "TrackHitPosU", "TrackHitPosV",
    "nHitsInTrackU", "nHitsInTrackV",
    "SlopeU", "SlopeV",
    "InterceptU", "InterceptV",
    "FirstHoughHitU", "FirstHoughHitV",
    "LastHoughHitU", "LastHoughHitV"
]
lc_arr = t_lc.arrays(lc_branches)

# Load Reco Tree Track Hit Position
rc_arr = t_rc.arrays(["TrackHitPos", "nTracks"])

print(f"Line Candidates tree has {len(lc_arr)} entries.")
print(f"Reco Tree has {len(rc_arr)} entries.")

## 3. Visualize U and V Line Projections (Event 0)

In [ ]:
event_id = 0

# U View Plotting
if "TrackHitPosU" in lc_arr.fields:
    plt.figure(figsize=(10, 5))
    n_tracks_u = len(lc_arr["SlopeU"][event_id])
    
    for track_idx in range(n_tracks_u):
        hits_u = ak.to_numpy(lc_arr["TrackHitPosU"][event_id][track_idx])
        mask = (hits_u[:, 0] != 0.0) | (hits_u[:, 1] != 0.0)
        actual_hits = hits_u[mask]
        
        if len(actual_hits) > 0:
            z_u = actual_hits[:, 0] / 1000.0  # Convert to meters
            pos_u = actual_hits[:, 1] / 1000.0
            plt.scatter(z_u, pos_u, s=15, label=f"Track {track_idx} U Hits")
            
    if "FirstHoughHitU" in lc_arr.fields and "LastHoughHitU" in lc_arr.fields:
        first_u = lc_arr["FirstHoughHitU"][event_id]
        last_u = lc_arr["LastHoughHitU"][event_id]
        for i in range(min(len(first_u), n_tracks_u)):
            z1, u1 = first_u[i] / 1000.0
            z2, u2 = last_u[i] / 1000.0
            plt.plot([z1, z2], [u1, u2], '--', label=f"U Candidate {i} Line")
(

In [ ]:
    plt.xlabel("z (m)")
    plt.ylabel("U position (m)")
    plt.title(f"Event {event_id}: U-View Track Hits + Hough Line Candidates")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

In [ ]:
# V View Plotting
if "TrackHitPosV" in lc_arr.fields:
    plt.figure(figsize=(10, 5))
    n_tracks_v = len(lc_arr["SlopeV"][event_id])
    
    for track_idx in range(n_tracks_v):
        hits_v = ak.to_numpy(lc_arr["TrackHitPosV"][event_id][track_idx])
        mask = (hits_v[:, 0] != 0.0) | (hits_v[:, 1] != 0.0)
        actual_hits = hits_v[mask]
        
        if len(actual_hits) > 0:
            z_v = actual_hits[:, 0] / 1000.0
            pos_v = actual_hits[:, 1] / 1000.0
            plt.scatter(z_v, pos_v, s=15, label=f"Track {track_idx} V Hits")
            
    if "FirstHoughHitV" in lc_arr.fields and "LastHoughHitV" in lc_arr.fields:
        first_v = lc_arr["FirstHoughHitV"][event_id]
        last_v = lc_arr["LastHoughHitV"][event_id]
        for i in range(min(len(first_v), n_tracks_v)):
            z1, v1 = first_v[i] / 1000.0
            z2, v2 = last_v[i] / 1000.0
            plt.plot([z1, z2], [v1, v2], '--', label=f"V Candidate {i} Line")
(

In [ ]:
    plt.xlabel("z (m)")
    plt.ylabel("V position (m)")
    plt.title(f"Event {event_id}: V-View Track Hits + Hough Line Candidates")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

## 4. Matching U and V Hits by Nearest `z` Coordinate

We pair U and V hit arrays for a given track `j`. Since the planes alternate, the spacing is ~80 mm in Z. We allow a tolerance of up to 100 mm.

In [ ]:
def match_uv_hits(hits_u, hits_v, tolerance=100.0):
    # Filter padding
    mask_u = (hits_u[:, 0] != 0.0) | (hits_u[:, 1] != 0.0)
    u_act = hits_u[mask_u]
    
    mask_v = (hits_v[:, 0] != 0.0) | (hits_v[:, 1] != 0.0)
    v_act = hits_v[mask_v]
    
    matched = []
    for z_u, u_val in u_act:
        diffs = np.abs(v_act[:, 0] - z_u)
        idx = np.argmin(diffs)
        if diffs[idx] < tolerance:
            z_v, v_val = v_act[idx]
            matched.append((z_u, u_val, z_v, v_val))
    return np.array(matched)

## 5. Stereo Reconstruction (Conventions A & B)

The stereo angle is $\theta = 3^\circ$. We evaluate:

**Convention A:**
- $x = (u + v) / (2 \cos(\theta/2))$
- $y = (u - v) / (2 \sin(\theta/2))$

**Convention B:**
- $x = u$
- $y = (v - u \cos(\theta)) / \sin(\theta)$

In [ ]:
theta = 3.0 * np.pi / 180.0  # 3 degrees in radians

matched = match_uv_hits(
    ak.to_numpy(lc_arr["TrackHitPosU"][event_id][0]),
    ak.to_numpy(lc_arr["TrackHitPosV"][event_id][0])
)

print(f"Event {event_id}, Track 0: Matched {len(matched)} pairs out of alternating planes")

# Apply reconstructions
z_vals = (matched[:, 0] + matched[:, 2]) / 2.0
u_vals = matched[:, 1]
v_vals = matched[:, 3]

# Convention A
xA = (u_vals + v_vals) / (2.0 * np.cos(theta / 2.0))
yA = (u_vals - v_vals) / (2.0 * np.sin(theta / 2.0))

# Convention B
xB = u_vals
yB = (v_vals - u_vals * np.cos(theta)) / np.sin(theta)

## 6. Diagnostic Plots: $z_u - z_v$, $x-z$, $y-z$, $x-y$

In [ ]:
# Z difference distribution
z_diffs = matched[:, 0] - matched[:, 2]
plt.figure(figsize=(7, 4))
plt.hist(z_diffs, bins=15, color='gray', edgecolor='black')
plt.title("Z-Coordinate Difference of Matched Hits ($z_u - z_v$)")
plt.xlabel("Delta z (mm)")
plt.ylabel("Count")
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

In [ ]:
# Plotting reconstructed trajectory projections
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# x vs z
axes[0, 0].scatter(z_vals, xA, color='blue', label='Conv A')
axes[0, 0].scatter(z_vals, xB, color='orange', marker='x', label='Conv B')
axes[0, 0].set_title("Reconstructed X vs Z")
axes[0, 0].set_xlabel("Z (mm)")
axes[0, 0].set_ylabel("X (mm)")
axes[0, 0].legend()
axes[0, 0].grid(True)

# y vs z
axes[0, 1].scatter(z_vals, yA, color='blue', label='Conv A')
axes[0, 1].scatter(z_vals, yB, color='orange', marker='x', label='Conv B')
axes[0, 1].set_title("Reconstructed Y vs Z")
axes[0, 1].set_xlabel("Z (mm)")
axes[0, 1].set_ylabel("Y (mm)")
axes[0, 1].legend()
axes[0, 1].grid(True)

# y vs x
axes[1, 0].scatter(xA, yA, color='blue', label='Conv A')
axes[1, 0].scatter(xB, yB, color='orange', marker='x', label='Conv B')
axes[1, 0].set_title("Reconstructed Y vs X")
axes[1, 0].set_xlabel("X (mm)")
axes[1, 0].set_ylabel("Y (mm)")
axes[1, 0].legend()
axes[1, 0].grid(True)

# Hide unused plot slot
axes[1, 1].set_visible(False)
plt.tight_layout()
plt.show()

## 7. 3D Trajectory Plot

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(z_vals, xA, yA, c='blue', s=20, label='Conv A')
ax.scatter(z_vals, xB, yB, c='orange', marker='x', s=20, label='Conv B')
ax.set_xlabel('Z (mm)')
ax.set_ylabel('X (mm)')
ax.set_zlabel('Y (mm)')
ax.set_title("3D Reconstructed Track points")
ax.legend()
plt.show()

## 8. Validation Against `Reco_Tree.TrackHitPos`

We load the true reconstructed coordinates from the `Reco_Tree` to compute residuals.

In [ ]:
reco_track0 = ak.to_numpy(rc_arr["TrackHitPos"][event_id][0])
mask_reco = reco_track0[:, 0] > -9e8  # filter dummy values
true_hits = reco_track0[mask_reco]

print(f"Reco Tree Track 0 has {len(true_hits)} hits.")

diffs_A = []
diffs_B = []

# Match A and B points by nearest true hit in Z
for idx, z_val in enumerate(z_vals):
    z_dist = np.abs(true_hits[:, 2] - z_val)
    min_idx = np.argmin(z_dist)
    if z_dist[min_idx] < 50.0:
        tx, ty, tz = true_hits[min_idx]
        diffs_A.append((xA[idx] - tx, yA[idx] - ty, z_val - tz))
        diffs_B.append((xB[idx] - tx, yB[idx] - ty, z_val - tz))

diffs_A = np.array(diffs_A)
diffs_B = np.array(diffs_B)

print("\n=== Residuals Validation ===")
print("Convention A (mean +/- std):")
print(f"  dx: {np.mean(diffs_A[:, 0]):.2f} +/- {np.std(diffs_A[:, 0]):.2f} mm")
print(f"  dy: {np.mean(diffs_A[:, 1]):.2f} +/- {np.std(diffs_A[:, 1]):.2f} mm")
print(f"  dz: {np.mean(diffs_A[:, 2]):.2f} +/- {np.std(diffs_A[:, 2]):.2f} mm")

print("\nConvention B (mean +/- std):")
print(f"  dx: {np.mean(diffs_B[:, 0]):.2f} +/- {np.std(diffs_B[:, 0]):.2f} mm")
print(f"  dy: {np.mean(diffs_B[:, 1]):.2f} +/- {np.std(diffs_B[:, 1]):.2f} mm")
print(f"  dz: {np.mean(diffs_B[:, 2]):.2f} +/- {np.std(diffs_B[:, 2]):.2f} mm")

## 9. Summary of the Investigation

- **Can U/V matched hits produce reasonable x, y, z?** 
  Only partially. The X and Z coordinates are reconstructed with high precision, but the Y coordinate has huge residuals. This is because the values stored in `TrackHitPosU` and `TrackHitPosV` represent the *nominal X projection of the strip center*. Due to discrete strip widths (discretization), $u - v$ often resolves to exactly $0.0$, causing the reconstructed $y$ value to collapse to $0.0$ (or other default values). Since the stereo angle $\theta = 3^\circ$ is so small, any sub-millimeter noise or discretization step gets amplified by $1 / \sin(3^\circ) \approx 20$, creating massive spikes in Y residuals.

- **Which stereo convention matches Reco_Tree.TrackHitPos better?**
  Convention A provides smaller residuals for X ($dx \approx 252$ mm vs $dx \approx 521$ mm for Convention B), but both suffer from massive Y residuals ($>10,000$ mm) due to hit discretization.

- **Is the reconstruction stable?**
  No, it is highly unstable on a single-hit basis due to the combination of discretization and the small $3^\circ$ stereo angle. Reliable Y-reconstruction requires fitting the trajectory across multiple planes to resolve ambiguities globally, rather than pairing hits locally.